In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

THREAD_URLS = [
    "https://voz.vn/t/chang-trai-%C4%90a-nang-ngo-ngang-khi-lan-%C4%91au-an-xoi-che-caramel-o-ha-noi.1196080/",
    "https://voz.vn/t/platform-grab-bee-etc-anh-em-thao-luan-vao-%C4%91ay.1117332/"
 ]

if not THREAD_URLS:
    print("Hãy điền các URL thread VOZ vào THREAD_URLS trước khi chạy crawl.")
else:
    print(f"Đã cấu hình {len(THREAD_URLS)} thread để crawl.")

Đã cấu hình 2 thread để crawl.


In [9]:
import random
import re

def setup_driver():
    """Khởi tạo undetected Chrome WebDriver để bypass detection"""
    options = uc.ChromeOptions()
    # options.add_argument("--headless=new")  # Headless mode mới hơn
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    
    # Tạo driver với undetected-chromedriver
    driver = uc.Chrome(options=options, version_main=None)
    
    # Set thêm một số properties để giống người dùng thật hơn
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        """
    })
    
    return driver

def fetch_html_selenium(driver, url, delay=3.0):
    """Lấy HTML từ URL bằng Selenium với các thao tác giống người dùng"""
    try:
        driver.get(url)
        # Đợi trang tải (đợi body xuất hiện)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Simulate human-like behavior
        time.sleep(random.uniform(1.5, 2.5))
        
        # Scroll từ từ như người dùng thật
        total_height = driver.execute_script("return document.body.scrollHeight")
        viewport_height = driver.execute_script("return window.innerHeight")
        current_position = 0
        
        while current_position < total_height:
            # Scroll một đoạn ngẫu nhiên
            scroll_by = random.randint(200, 400)
            driver.execute_script(f"window.scrollBy(0, {scroll_by});")
            current_position += scroll_by
            time.sleep(random.uniform(0.2, 0.5))
        
        # Scroll về đầu trang
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(random.uniform(1, 2))
        
        return driver.page_source
    except Exception as e:
        print(f"Lỗi khi tải {url}: {e}")
        return None

def clean_text(text):
    """Làm sạch text, loại bỏ metadata không cần thiết"""
    # Loại bỏ "Reactions: ..." 
    text = re.sub(r'\s*Reactions:.*$', '', text, flags=re.MULTILINE)
    
    # Loại bỏ "via theNEXTvoz for iPhone/Android"
    text = re.sub(r'\s*via theNEXTvoz.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*via.*for (iPhone|Android).*$', '', text, flags=re.MULTILINE)
    
    # Loại bỏ timestamp và post number ở đầu (format: "Jan 12, 2026 #10")
    text = re.sub(r'^\s*[A-Z][a-z]{2}\s+\d{1,2},\s+\d{4}\s+#\d+\s*', '', text, flags=re.MULTILINE)
    
    # Loại bỏ rank/badge (Senior Member, Đã tốn tiền, etc) - chỉ giữ nội dung sau
    # Pattern: bắt đầu với username, sau đó là rank, rồi là content
    text = re.sub(r'^[^\s]+\s+(Senior Member|Đã tốn tiền|Junior Member|Member|Banned)\s+', '', text, flags=re.MULTILINE)
    
    return text.strip()

def parse_thread_page(html, thread_url):
    soup = BeautifulSoup(html, "html.parser")
    posts_data = []
    
    # Trong XenForo, mỗi post thường là <article class="message" ...>
    articles = soup.select("article.message")
    print(f"  → Tìm thấy {len(articles)} articles trên trang")
    
    for idx, article in enumerate(articles, start=1):
        # Username
        author_tag = article.select_one("h4.message-name a, a.username")
        author = author_tag.get_text(strip=True) if author_tag else None
        
        # Thời gian
        time_tag = article.select_one("time")
        created_at = time_tag["datetime"] if time_tag and time_tag.has_attr("datetime") else (time_tag.get_text(strip=True) if time_tag else None)
        
        # Nội dung text - chỉ lấy từ bbWrapper để tránh lấy metadata
        content_tag = article.select_one("div.bbWrapper")
        if not content_tag:
            content_tag = article.select_one("div.message-body")
        if not content_tag:
            content_tag = article
            
        # Loại bỏ các elements không cần thiết
        for unwanted in content_tag.select("blockquote, .message-signature, .reactionsBar, .message-lastEdit"):
            unwanted.decompose()
        
        text = content_tag.get_text(" ", strip=True)
        
        # Làm sạch text
        text = clean_text(text)
        
        if text:  # Chỉ thêm nếu có nội dung
            posts_data.append({
                "thread_url": thread_url,
                "post_index": idx,
                "author": author,
                "created_at": created_at,
                "text": text,
            })
    
    # Tìm link trang kế tiếp (nếu có)
    next_link = None
    next_tag = soup.select_one("a.pageNav-jump--next, a[rel='next']")
    if next_tag and next_tag.has_attr("href"):
        href = next_tag["href"]
        if href.startswith("http"):
            next_link = href
        else:
            # xử lý link tương đối
            if href.startswith("/"):
                base = "https://voz.vn"
                next_link = base + href
            else:
                # trường hợp hiếm
                next_link = href
    
    return posts_data, next_link

def crawl_thread(driver, thread_url, max_pages=None):
    all_posts = []
    url = thread_url
    page_count = 0
    while url:
        print(f"Đang crawl: {url}")
        html = fetch_html_selenium(driver, url)
        if not html:
            break
        posts, next_url = parse_thread_page(html, thread_url)
        all_posts.extend(posts)
        page_count += 1
        if max_pages is not None and page_count >= max_pages:
            break
        url = next_url
    print(f"✓ Thread {thread_url} - tổng số post: {len(all_posts)} (trên {page_count} trang)")
    return all_posts

In [10]:
all_threads_posts = []

# Khởi tạo driver
driver = setup_driver()

try:
    for i, t_url in enumerate(THREAD_URLS, 1):
        print(f"\n[{i}/{len(THREAD_URLS)}] Bắt đầu crawl thread...")
        posts = crawl_thread(driver, t_url, max_pages=5)  # có thể đổi max_pages nếu muốn giới hạn
        all_threads_posts.extend(posts)
        
        # Delay giữa các thread để tránh bị block
        if i < len(THREAD_URLS):
            wait_time = random.uniform(3, 6)
            print(f"  → Chờ {wait_time:.1f}s trước khi crawl thread tiếp theo...")
            time.sleep(wait_time)
finally:
    # Đóng driver sau khi hoàn thành
    driver.quit()
    print("\n✓ Đã đóng browser")

df = pd.DataFrame(all_threads_posts)
print(f"\n📊 Tổng số post lấy được: {len(df)}")
if not df.empty:
    print(df.head())
    print(f"\nPhân bố theo thread:")
    print(df.groupby("thread_url").size())


[1/2] Bắt đầu crawl thread...
Đang crawl: https://voz.vn/t/chang-trai-%C4%90a-nang-ngo-ngang-khi-lan-%C4%91au-an-xoi-che-caramel-o-ha-noi.1196080/
  → Tìm thấy 20 articles trên trang
Đang crawl: https://voz.vn/t/chang-trai-%C4%90a-nang-ngo-ngang-khi-lan-%C4%91au-an-xoi-che-caramel-o-ha-noi.1196080/page-2
  → Tìm thấy 1 articles trên trang
✓ Thread https://voz.vn/t/chang-trai-%C4%90a-nang-ngo-ngang-khi-lan-%C4%91au-an-xoi-che-caramel-o-ha-noi.1196080/ - tổng số post: 21 (trên 2 trang)
  → Chờ 5.5s trước khi crawl thread tiếp theo...

[2/2] Bắt đầu crawl thread...
Đang crawl: https://voz.vn/t/platform-grab-bee-etc-anh-em-thao-luan-vao-%C4%91ay.1117332/
  → Tìm thấy 20 articles trên trang
Đang crawl: https://voz.vn/t/platform-grab-bee-etc-anh-em-thao-luan-vao-%C4%91ay.1117332/page-2
  → Tìm thấy 20 articles trên trang
Đang crawl: https://voz.vn/t/platform-grab-bee-etc-anh-em-thao-luan-vao-%C4%91ay.1117332/page-3
  → Tìm thấy 20 articles trên trang
Đang crawl: https://voz.vn/t/platform-gr

In [11]:
if not df.empty:
    df.to_csv("voz_threads_comments.csv", index=False, encoding="utf-8-sig")
    print("✓ Đã lưu voz_threads_comments.csv")
    
    text_corpus = df["text"].dropna().astype(str)
    with open("voz_text_corpus.txt", "w", encoding="utf-8") as f:
        for line in text_corpus:
            line = line.replace("\n", " ").strip()
            if line:
                f.write(line + "\n")
    print("✓ Đã lưu voz_text_corpus.txt")
else:
    print("Không có dữ liệu để lưu.")

✓ Đã lưu voz_threads_comments.csv
✓ Đã lưu voz_text_corpus.txt
